# C1. Fixed gene panel from a TCGA discovery split

The earlier notebooks chose genes inside each resample. That is fine in simulation, but on hardware it would mean
new circuits for every resample. Here the genes are chosen **once**, on TCGA tumors that are then set aside.

| Set | Tumors | Used for |
|---|---|---|
| TCGA discovery | about 410 | ranking the genes. Never measured, never scored. |
| TCGA measured | the remaining ~400 | train/test resampling, done classically after one measurement |
| METABRIC measured | 600 | independent validation |

The panel is nested: an n-qubit run uses the top n genes. Everything is saved to `credits/outputs/fixed_panel.npz`,
which notebooks C2 and C3 read. Run time: about 2 minutes.

In [1]:
TASK        = "TP53"
N_DISCOVERY = 410      # TCGA tumors used only to choose the genes
N_METABRIC  = 600      # METABRIC tumors in the measured set
N_PANEL     = 156      # genes in the panel = qubits at full width
N_PREFILTER = 5000     # most variable genes kept before ranking (no labels used)
SEED        = 2026     # fixes the discovery split; change it only on purpose

In [2]:
import json, time, math, warnings
import os, sys
from pathlib import Path
# Find the project folder (the one holding the pqk100 package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd(), Path(r"D:\Active Research Projects\PQK100")]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "pqk100" / "__init__.py").exists() and (p / "credits").is_dir():
                return p.resolve()
    raise RuntimeError(f"Cannot find the PQK100 project folder from {Path.cwd()}. "
                       "Set ROOT = Path(r'D:\\Active Research Projects\\PQK100') by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project folder:", ROOT)
OUT = ROOT / "credits" / "outputs"; OUT.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
pd.set_option("display.width", 200)

from pqk100.data import load_cohorts
from pqk100.prep import _strat_pick, select_genes, make_split

project folder: D:\Active Research Projects\PQK100


## C1.1 Cohorts

In [3]:
tcga, meta = load_cohorts()
ya_all, yb_all = tcga.labels[TASK].values, meta.labels[TASK].values
A, Bi = np.flatnonzero(~np.isnan(ya_all)), np.flatnonzero(~np.isnan(yb_all))
ya, yb = ya_all[A].astype(int), yb_all[Bi].astype(int)
print(f"TCGA-BRCA: {len(A)} tumors with a {TASK} call, {ya.mean():.1%} mutated")
print(f"METABRIC : {len(Bi)} tumors with a {TASK} call, {yb.mean():.1%} mutated")

TCGA-BRCA: 811 tumors with a TP53 call, 34.4% mutated
METABRIC : 1904 tumors with a TP53 call, 34.6% mutated


## C1.2 Split

In [4]:
rng  = np.random.default_rng(SEED)
disc = _strat_pick(ya, N_DISCOVERY, rng)                 # positions within the labeled TCGA tumors
meas = np.setdiff1d(np.arange(len(ya)), disc)           # everything else in TCGA is measured
ext  = _strat_pick(yb, N_METABRIC, rng)
assert not set(disc) & set(meas)

pd.DataFrame([
    ("TCGA discovery", len(disc), 100 * ya[disc].mean(), "choose genes only"),
    ("TCGA measured",  len(meas), 100 * ya[meas].mean(), "train / held-out test"),
    ("METABRIC measured", len(ext), 100 * yb[ext].mean(), "independent validation"),
], columns=["set", "tumors", "mutated (%)", "role"]).round(1)

,set,tumors,mutated (%),role
0,TCGA discovery,410,34.4,choose genes only
1,TCGA measured,401,34.4,train / held-out test
2,METABRIC measured,600,34.5,independent validation


## C1.3 Scaling and gene ranking

Each gene is z-scored within its own cohort over all labeled tumors, as in the earlier notebooks. That uses no labels.
The variance prefilter and the ANOVA ranking see the discovery tumors only.

In [5]:
Ra, Rb = tcga.X.values, meta.X.values
raw_var = Ra[A[disc]].astype(np.float64).var(0)
keep = np.argsort(-raw_var)[:N_PREFILTER]
Xa = Ra[A][:, keep].astype(np.float64); Xa = (Xa - Xa.mean(0)) / (Xa.std(0) + 1e-9)
Xb = Rb[Bi][:, keep].astype(np.float64); Xb = (Xb - Xb.mean(0)) / (Xb.std(0) + 1e-9)

rank  = select_genes(Xa[disc], ya[disc], N_PANEL)      # best first, near-duplicates skipped
genes = tcga.X.columns.values[keep][rank]
print(f"{len(genes)} genes. Top 20:")
print(", ".join(genes[:20]))

156 genes. Top 20:
CBX2, PSAT1, SCUBE2, AGR3, A2ML1, DNMT3B, CENPW, LRP8, TBC1D9, DNAJC12, CENPA, ARSG, SRD5A1, CCNE1, DNALI1, SPEF2, CAPN8, CDCA7, NME5, MLPH


## C1.4 Save

In [6]:
np.savez_compressed(
    OUT / "fixed_panel.npz",
    genes=genes,
    Z_tcga=Xa[meas][:, rank].astype(np.float32), y_tcga=ya[meas],
    ids_tcga=tcga.X.index.values[A[meas]].astype(str),
    Z_meta=Xb[ext][:, rank].astype(np.float32), y_meta=yb[ext],
    ids_meta=meta.X.index.values[Bi[ext]].astype(str),
    ids_discovery=tcga.X.index.values[A[disc]].astype(str),
)
info = {"task": TASK, "seed": SEED, "n_discovery": int(len(disc)), "n_tcga_measured": int(len(meas)),
        "n_metabric": int(len(ext)), "n_prefilter": N_PREFILTER, "genes": [str(g) for g in genes],
        "created": time.strftime("%Y-%m-%d %H:%M")}
(OUT / "fixed_panel.json").write_text(json.dumps(info, indent=1))
print("wrote", OUT / "fixed_panel.npz")
print(f"measured tumors: {len(meas)} TCGA + {len(ext)} METABRIC = {len(meas) + len(ext)}")

wrote D:\Active Research Projects\PQK100\credits\outputs\fixed_panel.npz
measured tumors: 401 TCGA + 600 METABRIC = 1001


## C1.5 How different is this panel from the per-resample panels?

Overlap with the genes the earlier sweep chose in each of its five resamples. A large overlap means the AUC values
should move little in C2.

In [7]:
old = []
for s in range(5):
    sp = make_split(tcga, meta, TASK, 200, 200, 400, s, "cohort")
    old.append(sp.genes[select_genes(sp.Xtr, sp.ytr, N_PANEL)])
rows = []
for n in (20, 60, 100, 156):
    ov = [len(set(o[:n]) & set(genes[:n])) / n for o in old]
    rows.append({"panel size": n, "mean overlap": f"{np.mean(ov):.0%}", "range": f"{min(ov):.0%} to {max(ov):.0%}"})
pd.DataFrame(rows)

,panel size,mean overlap,range
0,20,41%,30% to 55%
1,60,57%,52% to 67%
2,100,60%,58% to 64%
3,156,64%,58% to 69%
